In [ ]:
!pip install -q langchain langchain-openai

In [ ]:
from IPython.display import Image
from google.colab import userdata
from langchain.messages import AIMessage, HumanMessage
from langchain_core.runnables import Runnable, RunnablePassthrough
from langchain_openai import ChatOpenAI
from pathlib import Path
from pydantic import SecretStr

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

def print_response(response: AIMessage):
    print(f"Response id: {response.id}")
    if response.usage_metadata is not None:
        input_tokens = response.usage_metadata.get("input_tokens", 0)
        cached_tokens = response.usage_metadata.get("input_token_details", {}).get("cache_read", 0)
        output_tokens = response.usage_metadata.get("output_tokens", 0)
        reasoning_tokens = response.usage_metadata.get("output_token_details", {}).get("reasoning", 0)

        print(f"Input tokens: {input_tokens} ({cached_tokens} cached); Output tokens: {output_tokens} ({reasoning_tokens} reasoning)")

    print()
    print(f"{'-' * 20} [Output] {'-' * 20}")
    print(response.text)

def display_graph(runnable: Runnable, output_png: Path) -> None:
    graph = runnable.get_graph()

    with output_png.open(mode="wb") as file:
        file.write(graph.draw_mermaid_png())

    display(Image(output_png, format="png"))

In [ ]:
openai_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="low")
chain = RunnablePassthrough(lambda _: print("Start model execution")) | openai_model | RunnablePassthrough(lambda _: print("Model execution finished")) | RunnablePassthrough(print_response)

In [ ]:
conversation = [HumanMessage("Hello! How are you?")]
result = chain.invoke(conversation)

In [ ]:
# NOTE: `result` contains the AIMessage generated by the model. It is preserved due to the use of `RunnablePassthrough`.
result

In [ ]:
display_graph(openai_model, Path("/content/model.png"))

In [ ]:
display_graph(chain, Path("/content/chain.png"))